# RLHF Clinical Red-Teaming — Pipeline Demo
**Audrey Tjokro · Stephen Dong · Niki Karanikola**

End-to-end demo of all three methods (`baseline` / `dpo` / `ppo`) followed by a test-split evaluation. Every run uses **drastically reduced** hyperparameters — just enough to exercise the full pipeline (data → models → train → eval → artifact sync) so you can sanity-check the wiring before launching paper-grade runs.

This is a **driver only**: no training logic lives here. Each section shells out to `python -m src <method> ...` with an `OVERRIDES` block that shrinks the run to a few minutes.

**Total expected wall-time on a Colab A100:** ~15–25 min for all four sections.

## 1. Mount Drive (optional — used as a local cache for HF + checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone repo at a specific commit/branch
Pin a SHA for paper-grade reproducibility. Pass `BRANCH = 'main'` for HEAD.

In [ ]:
REPO_URL = 'https://github.com/stephendongg/rlhf-clinical-redteaming.git'
BRANCH   = 'combined'
REPO_DIR = '/content/rlhf-clinical-redteaming'

import os, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)
print(subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD']).decode().strip())
%cd $REPO_DIR

## 3. Install requirements

In [ ]:
%pip install -q -r requirements.txt
%pip install -q -e .

## 4. Secrets + GCS auth
Add `OPENAI_API_KEY` to Colab Secrets (left sidebar → key icon). The demo overrides each run to use the OpenAI judge (`gpt-4o-mini`) for speed.

In [ ]:
from google.colab import userdata, auth
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY').strip()

# GCS auth — uses your Google account; bucket gs://results_043026 must be in project rlhf-clinical-redteaming.
auth.authenticate_user()
os.environ['GOOGLE_CLOUD_PROJECT'] = 'rlhf-clinical-redteaming'

## 5. Shared demo settings + run helper

`TINY_DATA` shrinks the seed splits to a handful of scenarios. `TINY_GEN` shortens conversations and generations. Each method below adds its own training-loop shrink overrides on top.

If you want to scale up, edit just these dicts (or the per-method `OVERRIDES`).

In [ ]:
GCS_BUCKET = 'gs://results_043026'

# Tiny seed splits — full configs default to 100/50/100.
TINY_DATA = [
    'data.n_train=4',
    'data.n_dev=4',
    'data.n_test=4',
]

# Shorter conversations + generations — full configs default to 5 turns / 256 tokens.
TINY_GEN = [
    'max_turns=2',
    'attacker_max_new_tokens=128',
    'target_max_new_tokens=128',
]

# Use the OpenAI judge for speed; HF judge adds ~5 GB VRAM and a model load.
DEMO_JUDGE = [
    'judge_backend=openai',
    'judge_model=gpt-4o-mini',
]

# Demo runs may be off the pinned SHA with local edits — allow dirty.
EXTRA_FLAGS = ['--allow-dirty']

In [ ]:
import shlex, subprocess, time

def run_method(method: str, run_name: str, overrides: list, use_test: bool = False) -> int:
    """Build and stream a `python -m src <method> ...` invocation."""
    cmd = ['python', '-m', 'src', method,
           '--config', f'configs/{method}.yaml',
           '--run-name', run_name,
           '--gcs-bucket', GCS_BUCKET]
    for o in overrides:
        cmd += ['--override', o]
    if use_test:
        cmd.append('--use-test')
    cmd += EXTRA_FLAGS

    print('$', ' '.join(shlex.quote(c) for c in cmd))
    print('=' * 80)
    start = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    elapsed = (time.time() - start) / 60
    print('=' * 80)
    print(f'Finished {method!r} run_name={run_name!r} rc={rc} in {elapsed:.1f} min')
    if rc != 0:
        raise RuntimeError(f'{method} run failed with return code {rc}')
    return rc

## 6. Baseline (dev split)

Untuned attacker vs. target on the dev split. No training; this is the floor that DPO / PPO are trying to beat. Reports ASR, TTF, and effectiveness for n=4 scenarios.

In [ ]:
run_method(
    method='baseline',
    run_name='demo-baseline-dev',
    overrides=TINY_DATA + TINY_GEN + DEMO_JUDGE,
)

## 7. DPO (1 outer iter, dev eval)

Iterative DPO with the smallest possible loop:
- 1 outer iteration (collect → train → eval) instead of 4
- 2 rollouts per seed instead of 4
- 1 epoch over cached pairs instead of 2
- gradient_accum=1

A LoRA adapter is saved under the run dir and synced to GCS.

In [ ]:
DPO_TINY = [
    'dpo.n_outer=1',
    'dpo.n_per_seed=2',
    'dpo.n_epochs=1',
    'dpo.grad_accum=1',
]

run_method(
    method='dpo',
    run_name='demo-dpo-dev',
    overrides=TINY_DATA + TINY_GEN + DEMO_JUDGE + DPO_TINY,
)

## 8. PPO (2 train steps, dev eval)

PPO with the smallest possible loop:
- 2 training steps instead of 100
- 2 conversations per update instead of 4
- gradient_accumulation_steps=1
- checkpoint_every=2 (so we get one checkpoint at the end)

Final-step adapter is saved under the run dir and synced to GCS.

In [ ]:
PPO_TINY = [
    'ppo.n_train_steps=2',
    'ppo.n_convos_per_update=2',
    'ppo.gradient_accumulation_steps=1',
    'ppo.checkpoint_every=2',
]

run_method(
    method='ppo',
    run_name='demo-ppo-dev',
    overrides=TINY_DATA + TINY_GEN + DEMO_JUDGE + PPO_TINY,
)

## 9. Test-split evaluation

Adding `--use-test` to any method flips the final eval from the dev split to the held-out test split. For `dpo` / `ppo` this re-runs training as well — that's how the system is wired (training and final eval happen in the same invocation). For paper-grade numbers you'd run each method **once** with `--use-test` directly.

Below we just demo the test-eval flag on `baseline` (no training, so it's fast).

In [ ]:
run_method(
    method='baseline',
    run_name='demo-baseline-test',
    overrides=TINY_DATA + TINY_GEN + DEMO_JUDGE,
    use_test=True,
)

If you want to test-evaluate the trained DPO / PPO attackers as part of this demo, uncomment below. Each will retrain with the tiny knobs and then eval on the test split.

In [ ]:
# run_method(
#     method='dpo',
#     run_name='demo-dpo-test',
#     overrides=TINY_DATA + TINY_GEN + DEMO_JUDGE + DPO_TINY,
#     use_test=True,
# )
#
# run_method(
#     method='ppo',
#     run_name='demo-ppo-test',
#     overrides=TINY_DATA + TINY_GEN + DEMO_JUDGE + PPO_TINY,
#     use_test=True,
# )

## 10. Inspect this demo's artifacts in GCS

In [ ]:
!gsutil ls -r {GCS_BUCKET}/baseline/ | grep demo- | tail -20
!gsutil ls -r {GCS_BUCKET}/dpo/      | grep demo- | tail -20
!gsutil ls -r {GCS_BUCKET}/ppo/      | grep demo- | tail -20